In [17]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import MarkdownHeaderTextSplitter
from langchain.text_splitter import CharacterTextSplitter

In [8]:
loader = PyPDFLoader("../../data/raw/essentials-of-general-surgery-and-surgical-specialties_compress.pdf")
pages = loader.load()
print(f"Number of pages = {len(pages)}")

Number of pages = 1403


In [16]:
page1 = pages [1000]
print(page1.page_content)

facial nerve. When the main trunk or one of the branches of the nerve is involved, paralysis of the muscles
supplied by that portion of the nerve may be seen. Transection of the nerve proximal to the lateral canthus
should be treated by primary reanastomosis or nerve grafting.
Facial paralysis significantly affects appearance; however, the most serious side effect is an ipsilateral
exposure keratitis caused by an inability to close the upper eyelid. If ignored, this keratitis may lead to corneal
scarring and even blindness. To protect the eye, the application of artificial tears, ointment, and tape (to keep
the eye closed while sleeping) is recommended. A tarsorrhaphy or the upper eyelid gold weight to allow better
coverage of the cornea may be required.
Otologic Neoplasms
Because the skin of the external ear receives as much exposure to ultraviolet (UV) light as skin on any other
area of the body, it is subject to the usual UV-light–induced skin neoplasms. Actinic keratosis is the mos

In [13]:
print(page1.metadata)

{'producer': 'calibre 3.39.1 [https://calibre-ebook.com]', 'creator': 'calibre (3.39.1) [https://calibre-ebook.com]', 'creationdate': '2019-03-09T22:06:18+00:00', 'author': 'Peter F. Lawrence', 'moddate': '2019-03-10T06:10:18+08:00', 'title': 'Essentials of General Surgery and Surgical Specialties', 'source': '../../data/raw/essentials-of-general-surgery-and-surgical-specialties_compress.pdf', 'total_pages': 1403, 'page': 10, 'page_label': '11'}


In [18]:
# Character splitter 
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)

In [19]:
docs = text_splitter.split_documents(pages)

In [22]:
docs[300].page_content

'thromboembolism (VTE). The evaluation of patients for surgery must include an assessment of the degree of\nrisk the patient has for a VTE event. Virtually all surgery carries varying degrees of risk for VTE, from\nminimal to highly significant. A number of steps are essential in the assessment of the degree of risk in a\npatient.\nTABLE 4-5\n Differential Diagnosis of Hypercoagulable States by Site of Thrombosis\nArterial Thrombosis (e.g., Myocardial Infarction)\nVenous Thrombosis (e.g., VTE)\nCommon: Antiphospholipid syndrome\nCommon: Factor V Leiden\n    Prothrombin 20210 mutation\n    Prothrombin 20210\n    HIT Syndrome\n    Protein C deficiency\nUncommon: Elevated PAI-1 activity\nProtein S deficiency\n    Hyperhomocysteinemia (strokes in children)\n    Antithrombin deficiency\n    tPA Deficiency\n    Uncommon: Hyperhomocysteinemia (uremic patients)\nAnomalous coronary arteries\n    Factor XII deficiency\nVasculitis\nTrauma\nImmobilization'

In [7]:
import os
import fitz  # pymupdf for PDF text extraction
import psycopg2
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings # or use SentenceTransformers
from langchain.vectorstores import Chroma , Pinecone
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import numpy as np

In [26]:
pdf_folder_path = "../../data/raw/"  # Update with your folder path
output_folder = "../../data/images"  # Update with your output folder path

In [27]:
#Extract images from the PDF and save them to the specified output folder
def extract_images_from_pdf(pdf_path:str, output_folder:str)-> list:
    """Extract images from a PDF file and save them to the specified output folder.
    Args:
        pdf_path (str): Path to the PDF file.
        output_folder (str): Path to the output folder where images will be saved.  "
        "Returns:"
        "list: List of paths to the saved images.
    """
    doc = fitz.open(pdf_path)
    image_paths = []

    for page_num in range(len(doc)):
        for img_index, img in enumerate(doc[page_num].get_images(full=True)):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            img_filename = f"{output_folder}/page_{page_num + 1}_img_{img_index + 1}.png"

            with open(img_filename, "wb") as img_file:
                img_file.write(image_bytes)
            
            image_paths.append(img_filename)

    return image_paths  # Return list of image file paths

In [28]:
import pdfplumber
import pandas as pd

def extract_tables_from_pdf(pdf_path):
    tables_list = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()
            for table in tables:
                df = pd.DataFrame(table)
                tables_list.append(df)
    
    return tables_list  # Return list of Pandas DataFrames

In [17]:
tables_list = extract_tables_from_pdf(pdf_folder_path)

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

In [21]:
tables_list[2]

,0
0,
1,C <3.0 >3.0 Severe Severe Poor >80


In [12]:
extract_images_from_pdf(pdf_folder_path, output_folder)

['../../data/images/page_1_img_1.png',
 '../../data/images/page_4_img_1.png',
 '../../data/images/page_60_img_1.png',
 '../../data/images/page_63_img_1.png',
 '../../data/images/page_65_img_1.png',
 '../../data/images/page_71_img_1.png',
 '../../data/images/page_95_img_1.png',
 '../../data/images/page_109_img_1.png',
 '../../data/images/page_132_img_1.png',
 '../../data/images/page_134_img_1.png',
 '../../data/images/page_134_img_2.png',
 '../../data/images/page_136_img_1.png',
 '../../data/images/page_136_img_2.png',
 '../../data/images/page_137_img_1.png',
 '../../data/images/page_138_img_1.png',
 '../../data/images/page_142_img_1.png',
 '../../data/images/page_150_img_1.png',
 '../../data/images/page_152_img_1.png',
 '../../data/images/page_153_img_1.png',
 '../../data/images/page_154_img_1.png',
 '../../data/images/page_173_img_1.png',
 '../../data/images/page_174_img_1.png',
 '../../data/images/page_180_img_1.png',
 '../../data/images/page_180_img_2.png',
 '../../data/images/page_

In [ ]:
"""from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model (you can use OpenAI, Cohere, etc.)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def semantic_chunking(text, chunk_size=500, chunk_overlap=100):
    """Splits text semantically based on embedding similarity."""
    # Split into sentences
    sentences = text.split(". ")
    embeddings = np.array([embedder.encode(s) for s in sentences])

    # Create chunks by grouping similar sentences
    chunks = []
    temp_chunk = []
    temp_embed = []
    
    for i, sentence in enumerate(sentences):
        temp_chunk.append(sentence)
        temp_embed.append(embeddings[i])

        if len(" ".join(temp_chunk)) > chunk_size:
            chunks.append(" ".join(temp_chunk))
            temp_chunk = temp_chunk[-(chunk_overlap // len(temp_chunk)):]  # Overlap
            temp_embed = temp_embed[-(chunk_overlap // len(temp_embed)):]
    
    if temp_chunk:
        chunks.append(" ".join(temp_chunk))
    
    return chunks
"""

In [49]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


client = QdrantClient("localhost", port=6333)  # Use cloud URL if needed

"""# 🔹 Create a collection (if it doesn't exist)
collection_name = "rag_collection"
if collection_name not in client.get_collections().collections:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)  # 384 for MiniLM
    )"""

def store_chunks_in_qdrant(chunks):
    embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # Prepare documents
    payloads = []
    vectors = []
    
    for i, chunk in enumerate(chunks):
        vector = embeddings_model.embed_query(chunk)
        payloads.append({"chunk_id": i, "text": chunk})
        vectors.append(np.array(vector))

    # Insert into Qdrant
    client.upload_collection(
        collection_name=collection_name,
        vectors=vectors,
        payload=payloads,
    )
    
    print("✅ Chunks stored in Qdrant!")


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# =============================
# 📌 1. Extract Text from PDFs
# =============================
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + "\n"
    return text


def store_raw_text_in_postgres(text, filename):
    conn = psycopg2.connect(database="rag_db", user="jihed", password="Jiji@jiji123", host="localhost", port="5432")
    cur = conn.cursor()
    
    cur.execute("CREATE TABLE IF NOT EXISTS documents (id SERIAL PRIMARY KEY, filename TEXT, content TEXT)")

    # ✅ Convert text to UTF-8 and replace problematic characters
    text = text.encode("utf-8", "ignore").decode("utf-8")

    cur.execute("INSERT INTO documents (filename, content) VALUES (%s, %s)", (filename, text))
    
    conn.commit()
    cur.close()
    conn.close()



def chunk_text(text):
    """splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = splitter.split_text(text)"""

    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
    chunks = splitter.split_text(text)# 🔹 Connect to Qdrant (Local or Cloud)
    return chunks


def store_chunks_in_chromadb(chunks):
    # Initialize ChromaDB and embeddings
    store_chunks_in_qdrant(chunks)
    

def process_pdfs(pdf_folder):
    for pdf_file in os.listdir(pdf_folder):
        if pdf_file.endswith(".pdf"):
            pdf_path = os.path.join(pdf_folder, pdf_file)
            
            print(f"🔍 Processing {pdf_file}...")
            text = extract_text_from_pdf(pdf_path)
            #store_raw_text_in_postgres(text, pdf_file)
            chunks = semantic_chunking(text)
            print(chunks)
            # vector_db = store_chunks_in_chromadb(chunks)

    
    print("✅ Pipeline completed!")
    return chunks

chunks = process_pdfs(pdf_folder_path)


🔍 Processing 1853.PMC2518358.pdf...
['Associations of Renal Vascular Resistance\nWith Albuminuria and Other\nMacroangiopathy in Type 2\nDiabetic Patients\nKUMIKO HAMANO, MD1\nAI NITTA, MD1\nTAKAYASU OHTAKE, MD, PHD2\nSHUZO KOBAYASHI, MD, PHD2\nOBJECTIVE — Albuminuria can be caused by endothelial dysfunction as a result of ischemic\nnephropathy rather than classic diabetic nephropathy We studied whether renal vascular resis-\ntance (resistive index [RI]) of the main renal arteries could be associated with albuminuria and\nfurther assessed the relationship between RI and aorta stiffness measured by brachial-ankle\npulse-wave velocity (baPWV).\nRESEARCH DESIGN AND METHODS — We consecutively studied 150 patients with\ntype 2 diabetes and the absence of clinically overt renal artery stenosis', 'Associations of Renal Vascular Resistance\nWith Albuminuria and Other\nMacroangiopathy in Type 2\nDiabetic Patients\nKUMIKO HAMANO, MD1\nAI NITTA, MD1\nTAKAYASU OHTAKE, MD, PHD2\nSHUZO KOBAYASHI, MD,

In [57]:
print(chunks[300])

Disseminated intravascular
coagulopathy is an ominous complication that can occur with hours of placenta abruption or amniotic fluid
embolization Because sensitization of Rh-negative women occurs with miniscule amounts of fetal Rh-
positive blood, all injured Rh-negative women should be considered for Rh-immunoglobulin therapy, unless
the injury is relatively minor and remote from the uterus.
THE GERIATRIC PATIENT
The aging of the American population will continue for many decades to present some special challenges to
surgeons The elderly have less reserve than their younger counterparts They are often on medications that
can distort physiologic responses (e.g., β-blockers) They are also often on medications that can impact the
response to surgery (e.g., warfarin or platelet aggregation–inhibiting agents) Their ability to negotiate
everyday activities may be impaired at baseline, perhaps because of sensory impairment, difficulty in
ambulation, or dementia One of the conundrums that ari